# MMM in Hex: curve design and model results

Import this notebook into Hex (Projects → Import → .ipynb). Cells marked **HEX INPUT** become Hex input parameters (sliders) so marketers can explore curves without touching code. Publish as an App to share.


In [ ]:
# Hex runs in a managed Python environment; install the model library once
!pip install -q pymc-marketing==1.3.0 plotly


In [ ]:
import numpy as np, pandas as pd, plotly.graph_objects as go

# Upload data/mmm_geo_weekly.csv to the Hex project (Files panel), or point to your warehouse table
df = pd.read_csv('mmm_geo_weekly.csv', parse_dates=['date'])
df.head()


## 1 · Curve design
Check the shape of carryover and diminishing returns before fitting. These values become priors.

In [ ]:
# HEX INPUT: replace each line with a Hex slider of the same name
alpha = 0.6      # carryover decay rate (0 - 0.95)
K = 120          # half-saturation point, weekly GRPs
s = 2.0          # shape: 1 = C-shape, >1 = S-shape (effective-frequency threshold)
ceiling = 0.9    # maximum response share


In [ ]:
def hill(x, K, s, ceiling):
    x = np.asarray(x, float)
    return ceiling * x**s / (x**s + K**s)

half_life = np.log(0.5) / np.log(alpha)
x = np.linspace(0, 4 * K, 300)
y = hill(x, K, s, ceiling)
marginal = np.gradient(y, x)
fig = go.Figure([go.Scatter(x=x, y=y, name='Response'),
                 go.Scatter(x=x, y=marginal / marginal.max() * ceiling, name='Marginal return (indexed)', line=dict(dash='dot'))])
fig.update_layout(title=f'Half-life {half_life:.1f} weeks · K={K} · s={s}', xaxis_title='Weekly adstocked GRPs')
fig


## 2 · Fit the model
A quick PyMC-Marketing fit (use the full scripts in `src/` for production-length sampling).

In [ ]:
from pymc_extras.prior import Prior
from pymc_marketing.mmm import MMM, GeometricAdstock, HillSaturation

CH = ['tv_spend', 'ooh_spend', 'social_spend', 'search_spend', 'tiktok_spend']
CTRL = ['trend', 'holiday_peak', 'holiday_dip', 'promo', 'unemployment']
mmm = MMM(date_column='date', channel_columns=CH, target_column='revenue', control_columns=CTRL,
          dims=('geo',), yearly_seasonality=2,
          adstock=GeometricAdstock(l_max=8, priors={'alpha': Prior('Beta', alpha=[3,1,1,1,1], beta=[2,3,3,6,3], dims='channel')}),
          saturation=HillSaturation())
mmm.fit(df[['date', 'geo', *CH, *CTRL]], df['revenue'], draws=300, tune=300, chains=2, progressbar=False)


## 3 · Model results: weekly decomposition

In [ ]:
post = mmm.idata.posterior
scale = float(mmm.get_scales_as_xarray()['target_scale'])
nat = lambda v: (post[v].mean(('chain', 'draw')) * scale).sum('geo')
ch = nat('channel_contribution').to_pandas()
ch.columns = [c.replace('_spend', '') for c in ch.columns]
ctrl = nat('control_contribution').to_pandas()
base = pd.Series(float((post['intercept_contribution'].mean(('chain','draw')) * scale).sum()), index=ch.index, name='Base')
season = nat('yearly_seasonality_contribution').to_pandas().rename('Seasonality')
decomp = pd.concat([base, season, ctrl, ch], axis=1)
actual = df.groupby('date')['revenue'].sum()
decomp['Residual'] = actual.values - decomp.sum(axis=1).values
fig = go.Figure([go.Bar(x=decomp.index, y=decomp[c], name=c) for c in decomp.columns])
fig.add_scatter(x=actual.index, y=actual.values, name='Actual', line=dict(color='black'))
fig.update_layout(barmode='relative', title='Weekly revenue decomposition')
fig
